# 02 - Historique des expériences Lmaana

> Notebook 2/3 du rapport final.

Ce notebook documente les expériences déjà terminées et explique le choix de Lmaana 2.1. Il ne lance aucun entraînement. Les commandes de reproduction HPC sont conservées dans les notebooks HPC 04 et 05.

In [25]:
from pathlib import Path
import json
import sys

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import pandas as pd
from IPython.display import Markdown, display
from scripts.plot_lmaana_2_1_results import TEST, VALIDATION, VERSION_HISTORY

OUTPUTS = PROJECT / 'outputs'
CONFIGS = PROJECT / 'configs'
SELECTION = OUTPUTS / 'lmaana_clean_selections/long.json'
print('Projet :', PROJECT)

Projet : /home/skiredj.abderrahman/badr/darija-ctc-lmaana


## 1. Deux protocoles d’évaluation

Les versions v0 à v2 utilisent Dataset13 et MoulSot original. V5 et Lmaana 2.1 utilisent Dataset13 et Lmaana clean. Le changement de labels et de composition interdit une comparaison absolue entre MoulSot et Lmaana clean.

In [26]:
history_rows = []
for protocol, values in VERSION_HISTORY.items():
    secondary = 'MoulSot' if 'MoulSot' in values else 'Lmaana clean'
    for index, model in enumerate(values['models']):
        history_rows.append({
            'protocol': protocol,
            'model': model,
            'Dataset13 WER': values['Dataset13'][index],
            f'{secondary} WER': values[secondary][index],
        })
display(pd.DataFrame(history_rows))

,protocol,model,Dataset13 WER,MoulSot WER,Lmaana clean WER
0,Original MoulSot protocol,Lmaana v0,44.1211,45.4075,NaN
1,Original MoulSot protocol,Lmaana v1,44.2228,45.2389,NaN
2,Original MoulSot protocol,Lmaana v2,44.2674,45.2140,NaN
3,Clean Lmaana protocol,Retained V5,44.2243,NaN,45.1385
4,Clean Lmaana protocol,Lmaana 2.1,44.0393,NaN,45.1690


## 2. Expériences principales

1. Le curriculum depuis OmniASR de base a amélioré le modèle public, mais est resté derrière V5.
2. Le pilote de récupération de 1 000 steps depuis V5 a montré une légère amélioration Dataset13 puis un plateau.
3. L’adaptation longue de 5 000 steps depuis V5, avec encodeur gelé pendant 500 steps, a produit le candidat final.

In [27]:
experiment_configs = [
    'lmaana_clean_stage1.yaml',
    'lmaana_clean_stage2.yaml',
    'lmaana_clean_v5_recovery.yaml',
    'lmaana_clean_v5_long.yaml',
]
rows = []
for name in experiment_configs:
    path = CONFIGS / name
    rows.append({'configuration': name, 'available': path.is_file(), 'path': str(path)})
display(pd.DataFrame(rows))

,configuration,available,path
0,lmaana_clean_stage1.yaml,True,/home/skiredj.abderrahman/badr/darija-ctc-lmaa...
1,lmaana_clean_stage2.yaml,True,/home/skiredj.abderrahman/badr/darija-ctc-lmaa...
2,lmaana_clean_v5_recovery.yaml,True,/home/skiredj.abderrahman/badr/darija-ctc-lmaa...
3,lmaana_clean_v5_long.yaml,True,/home/skiredj.abderrahman/badr/darija-ctc-lmaa...


## 3. Progression de l’adaptation longue

Le step 0 correspond au checkpoint V5 conservé. La sélection compare ensuite les steps 3 500 et 5 000 sur les deux validations.

In [28]:
validation_rows = []
for step, datasets in VALIDATION.items():
    label = 'Retained V5' if step == 0 else f'Step {step}'
    for dataset, metrics in datasets.items():
        validation_rows.append({
            'step': step, 'model': label, 'dataset': dataset,
            'CER/UER': metrics['cer'], 'WER': metrics['wer'],
        })
validation_table = pd.DataFrame(validation_rows)
display(validation_table)

,step,model,dataset,CER/UER,WER
0,0,Retained V5,Dataset13,17.3908,43.2622
1,0,Retained V5,Lmaana clean,13.1994,40.4155
2,3500,Step 3500,Dataset13,17.3347,43.1245
3,3500,Step 3500,Lmaana clean,13.2053,40.4015
4,5000,Step 5000,Dataset13,17.3252,43.1238
5,5000,Step 5000,Lmaana clean,13.1925,40.3535


## 4. Règle de sélection

La sélection est faite uniquement sur validation. Dataset13 CER/UER puis WER sont prioritaires, avec des garde-fous de non-régression sur Lmaana clean. Le test n’est exécuté qu’une fois après la sélection.

In [29]:
if SELECTION.is_file():
    selection = json.loads(SELECTION.read_text(encoding='utf-8'))
    display(selection)
else:
    print('Sélection HPC absente localement :', SELECTION)
    display(Markdown('**Résultat enregistré : step 5 000 sélectionné.**'))

{'stage': 'long',
 'cer_tolerance_points': 0.05,
 'criterion': ['dataset13.uer',
  'dataset13.wer',
  'lmaana_clean.uer',
  'lmaana_clean.wer'],
 'selected': {'step': 5000,
  'checkpoint': '/home/skiredj.abderrahman/badr/darija-ctc-lmaana/outputs/lmaana_clean_v5_long/ws_1.bd2f3081/checkpoints/step_5000/model/pp_00/tp_00/sdp_00.pt',
  'score': [17.3252, 43.1238, 13.1925, 40.3535],
  'tests': {'dataset13': {'ctc_loss': 175.837,
    'uer': 17.3252,
    'wer': 43.1238,
    'examples': 6893},
   'lmaana_clean': {'ctc_loss': 75.163,
    'uer': 13.1925,
    'wer': 40.3535,
    'examples': 5187}},
  'summary': '/home/skiredj.abderrahman/badr/darija-ctc-lmaana/outputs/checkpoint_evaluations/lmaana_clean_long_step_5000_validation/summary.json'},
 'ranking': [{'step': 5000,
   'checkpoint': '/home/skiredj.abderrahman/badr/darija-ctc-lmaana/outputs/lmaana_clean_v5_long/ws_1.bd2f3081/checkpoints/step_5000/model/pp_00/tp_00/sdp_00.pt',
   'score': [17.3252, 43.1238, 13.1925, 40.3535],
   'tests': {'

## 5. Comparaison test V5 contre Lmaana 2.1

In [30]:
test_rows = []
for model, datasets in TEST.items():
    for dataset, metrics in datasets.items():
        test_rows.append({
            'model': model, 'dataset': dataset,
            'CER/UER': metrics['cer'], 'WER': metrics['wer'],
        })
test_table = pd.DataFrame(test_rows)
display(test_table)

pivot = test_table.pivot(index='dataset', columns='model', values=['CER/UER', 'WER'])
deltas = pd.DataFrame(index=pivot.index)
deltas['CER/UER delta'] = pivot[('CER/UER', 'Lmaana 2.1')] - pivot[('CER/UER', 'Retained V5')]
deltas['WER delta'] = pivot[('WER', 'Lmaana 2.1')] - pivot[('WER', 'Retained V5')]
display(deltas.style.format('{:+.4f}'))

,model,dataset,CER/UER,WER
0,Retained V5,Dataset13,17.7674,44.2243
1,Retained V5,Lmaana clean,14.6099,45.1385
2,Lmaana 2.1,Dataset13,17.7029,44.0393
3,Lmaana 2.1,Lmaana clean,14.6048,45.1690


,CER/UER delta,WER delta
dataset,,
Dataset13,-0.0645,-0.1850
Lmaana clean,-0.0051,+0.0305


## Conclusion

Lmaana 2.1 est une adaptation incrémentale et prudente de V5, pas un entraînement depuis zéro. Le gain Dataset13 est réel mais faible; Lmaana clean reste pratiquement stable. Le notebook 03 présente les figures, les exemples de transcription et l’analyse finale.